<a href="https://colab.research.google.com/github/hilalrusydin95/Hilal/blob/main/muh_hilal_rusydin_hidayat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# REGRESI LINEAR BERGANDA
# ============================================================
!pip install -q ipywidgets statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline

# ------------------------------------------------------------
# 1. MEMUAT DATASET
# ------------------------------------------------------------
url = "https://raw.githubusercontent.com/Alam-A99/DATAAKSI/refs/heads/main/dataset_fiskal_97_hipoFull.csv"
df = pd.read_csv(url)
print("Dataset berhasil dimuat!", df.shape)

daftar_kota = sorted(df["Kota"].unique())
daftar_variabel = (
    df.select_dtypes(include=[np.number]).drop(columns=["Tahun"]).columns.tolist()
)


def pilih_default(nama, cadangan):
    """Pakai nama kolom yang diinginkan jika ada, jika tidak pakai kolom ke-n."""
    return nama if nama in daftar_variabel else daftar_variabel[cadangan]


# ------------------------------------------------------------
# 2. WIDGET
# ------------------------------------------------------------
lebar = widgets.Layout(width="450px")
gaya = {"description_width": "130px"}

kota_dd = widgets.Dropdown(
    options=daftar_kota,
    value="Kota Makassar" if "Kota Makassar" in daftar_kota else daftar_kota[0],
    description="Pilih Kota:", layout=lebar, style=gaya,
)
y_dd = widgets.Dropdown(
    options=daftar_variabel, value=pilih_default("DBH_PAJAK", 0),
    description="Variabel Y (Dep):", layout=lebar, style=gaya,
)
x1_dd = widgets.Dropdown(
    options=daftar_variabel, value=pilih_default("bm_pendidikan", 1),
    description="Variabel X₁:", layout=lebar, style=gaya,
)
x2_dd = widgets.Dropdown(
    options=daftar_variabel, value=pilih_default("KETIMPANGAN", 2),
    description="Variabel X₂:", layout=lebar, style=gaya,
)
tombol = widgets.Button(
    description="Jalankan Analisis, Grafik 3D & Interpretasi",
    button_style="success", icon="check", layout=lebar,
)
output_area = widgets.Output()


# ------------------------------------------------------------
# 3. FUNGSI ANALISIS
# ------------------------------------------------------------
def jalankan_analisis(b):
    with output_area:
        clear_output(wait=True)

        kota = kota_dd.value
        var_y, var_x1, var_x2 = y_dd.value, x1_dd.value, x2_dd.value

        # Validasi input
        if len({var_y, var_x1, var_x2}) < 3:
            print("⚠️ Variabel Y, X₁, dan X₂ harus berbeda satu sama lain!")
            return

        data = df[df["Kota"] == kota][["Tahun", var_y, var_x1, var_x2]].dropna()
        if len(data) < 5:
            print(f"⚠️ Data {kota} terlalu sedikit ({len(data)} baris) untuk regresi berganda.")
            return

        # Model OLS
        X = sm.add_constant(data[[var_x1, var_x2]])
        hasil = sm.OLS(data[var_y], X).fit()

        b0 = hasil.params["const"]
        b1 = hasil.params[var_x1]
        b2 = hasil.params[var_x2]
        p1 = hasil.pvalues[var_x1]
        p2 = hasil.pvalues[var_x2]
        r2 = hasil.rsquared

        # --- Ringkasan statistik ---
        print("=" * 75)
        print(f" 📊 HASIL OLS REGRESI BERGANDA | KOTA: {kota.upper()}")
        print("=" * 75)
        print(hasil.summary())

        # --- Interpretasi otomatis ---
        sig1 = "signifikan" if p1 < 0.05 else "tidak signifikan secara statistik"
        sig2 = "signifikan" if p2 < 0.05 else "tidak signifikan secara statistik"
        tanda1 = "<" if p1 < 0.05 else ">"
        tanda2 = "<" if p2 < 0.05 else ">"

        print("\n" + "=" * 75)
        print("INTERPRETASI HASIL REGRESI")
        print("=" * 75)
        print("1. Persamaan Regresi:")
        print(f"   Ŷ = {b0:.3f} + ({b1:.3f} × {var_x1}) + ({b2:.3f} × {var_x2})")
        print(f"   • Konstanta ({b0:.3f}): jika {var_x1} dan {var_x2} bernilai 0, "
              f"estimasi {var_y} adalah {b0:.3f}.")
        print(f"   • Koefisien {var_x1} ({b1:.3f}): setiap kenaikan 1 satuan akan mengubah "
              f"{var_y} sebesar {b1:.3f} satuan (dengan asumsi {var_x2} konstan).")
        print(f"   • Koefisien {var_x2} ({b2:.3f}): setiap kenaikan 1 satuan akan mengubah "
              f"{var_y} sebesar {b2:.3f} satuan (dengan asumsi {var_x1} konstan).\n")

        print("2. Uji Signifikansi (taraf nyata 5% / 0.05):")
        print(f"   • {var_x1} (p-value = {p1:.4f}): berpengaruh {sig1} terhadap "
              f"{var_y} (karena p-value {tanda1} 0.05).")
        print(f"   • {var_x2} (p-value = {p2:.4f}): berpengaruh {sig2} terhadap "
              f"{var_y} (karena p-value {tanda2} 0.05).\n")

        print("3. Koefisien Determinasi (R²):")
        print(f"   • R² = {r2:.4f} ({r2*100:.2f}%). Artinya {var_x1} dan {var_x2} mampu "
              f"menjelaskan variasi {var_y} sebesar {r2*100:.2f}%, sedangkan "
              f"{(1-r2)*100:.2f}% sisanya dipengaruhi faktor lain di luar model.")
        print("=" * 75 + "\n")

        # --- Grafik 3D ---
        x1_vals = np.linspace(data[var_x1].min(), data[var_x1].max(), 30)
        x2_vals = np.linspace(data[var_x2].min(), data[var_x2].max(), 30)
        X1g, X2g = np.meshgrid(x1_vals, x2_vals)
        Yg = b0 + b1 * X1g + b2 * X2g

        fig = plt.figure(figsize=(11, 8))
        ax = fig.add_subplot(111, projection="3d")
        ax.plot_surface(X1g, X2g, Yg, alpha=0.5, cmap="viridis")
        ax.scatter(data[var_x1], data[var_x2], data[var_y],
                   color="royalblue", s=45, label="Data Aktual")
        for _, r in data.iterrows():
            ax.text(r[var_x1], r[var_x2], r[var_y], str(int(r["Tahun"])), fontsize=8)

        ax.set_xlabel(var_x1, labelpad=10)
        ax.set_ylabel(var_x2, labelpad=10)
        ax.set_zlabel(var_y, labelpad=10)
        ax.set_title(f"Regresi 3D: {var_y} vs {var_x1} & {var_x2}\n"
                     f"Kota: {kota} | R² = {r2:.3f}")
        ax.view_init(elev=25, azim=-60)
        ax.legend()
        plt.show()


tombol.on_click(jalankan_analisis)

# ------------------------------------------------------------
# 4. TAMPILKAN PANEL KONTROL
# ------------------------------------------------------------
ui = widgets.VBox([
    widgets.HTML("<h3>📊 Panel Kontrol Regresi Linear Berganda & Grafik 3D</h3>"),
    kota_dd, y_dd, x1_dd, x2_dd, tombol,
])
display(ui, output_area)

Dataset berhasil dimuat! (1067, 24)


Output()